# Analyse de sentiments sur news financieres avec FinBERT


## 0. Installation

In [ ]:
!pip install transformers
!pip install datasets
!pip install scikit-learn
!pip install matplotlib
!pip install seaborn
!pip install torch
!pip install pandas
!pip install numpy
!pip install evaluate

## 1. Importation et nettoyage des données 

In [ ]:
from datasets import load_dataset

# Chargement des données
dataset = load_dataset("FinanceMTEB/financial_phrasebank")

# Jeux de données
df_train = dataset["train"]
df_test = dataset["test"]

print(dataset)

# Valeurs manquantes
print(f"Valeurs manquantes : {sum(x is None for x in df_train['text'])}")

# Doublons

def count_duplicates(dataset, column):
    values = dataset[column]
    return len(values) - len(set(values))

print(f"Doublons : {count_duplicates(df_train,'text')}")

# Suppression des doublons
seen = set()

def is_unique(example):
    if example["text"] in seen:
        return False
    seen.add(example["text"])
    return True

df_train = df_train.filter(is_unique)

print(f"Doublons après suppression : {count_duplicates(df_train, 'text')}")

Le dataset utilisé contient 2264 phrases prises dans des news financieres. Il n'y a pas de phrases manquantes et il ne possede qu'un seul doublon que nous avons retiré

## 2. Analyse exploratoire des données

In [ ]:
import matplotlib.pyplot as plt
neu = neg = pos = 0
for i in df_train["label"]:
  if i == 0:
    neg = neg + 1
  elif i == 1:
    neu = neu + 1
  else:
    pos = pos + 1

categories = ['Neutre', 'Négatif', 'Positif']
counts = [neu, neg, pos] # Utilisez vos variables neu, neg, pos ici

plt.figure(figsize=(8, 5))
plt.bar(categories, counts, color=['skyblue', 'lightcoral', 'lightgreen'])
plt.xlabel('Sentiment')
plt.ylabel('Nombre de phrases')
plt.title('Distribution des sentiments')
plt.show()




La distribution des classes de notre dataset est plutot déséquilibrées avec une forte presence de neutre comparé aux deux autres classes. Dans un telle situation l'usage de l'accuracy comme metrque d'evaluation n'est pas optimal.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from transformers import AutoTokenizer
checkpoint = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# la longueur de chaque séquence de tokens
token_lengths = [len(tokenizer(text)["input_ids"]) for text in df_train["text"]]

# histogramme des longueurs de tokens
plt.figure(figsize=(10, 6))
plt.hist(token_lengths, bins=20, edgecolor='black', alpha=0.7)
plt.title('Distribution des longueurs de tokens')
plt.xlabel('Longueur des tokens')
plt.ylabel('Nombre de phrases')
plt.grid(axis='y', alpha=0.75)
plt.show()

import numpy as np

print(f"Moyenne : {np.mean(token_lengths):.2f}")
print(f"Médiane : {np.median(token_lengths):.2f}")
print(f"Maximum : {np.max(token_lengths)}")
print(f"95e percentile : {np.percentile(token_lengths,95):.0f}")

In [ ]:
import pandas as pd
import seaborn as sns

# Créer un DataFrame pour les analyses plus simple pour les manipulations
analysis_df = pd.DataFrame({
    'token_length': token_lengths,
    'label_text': df_train['label_text']
})

# Calcul des statistiques par sentiment
sentiment_stats = analysis_df.groupby('label_text')['token_length'].agg(['mean', 'median', 'max'])

display(sentiment_stats)

import seaborn as sns

plt.figure(figsize=(8,5))
sns.boxplot(data=analysis_df,
            x="label_text",
            y="token_length")
plt.show()

La taille moyenne des tokens est plutot proche entre les différentes classes on voulait identifier un potentiel patterns au niveau des tailles pour distinguer les classes mais ce n'et pas le cas. De plus on constate que la sequence la plus longue contient 150 tokens or les modeles BERT peuvent gerer jusqu'a 512 tokens par sequences l'utilisation dela truncation ne sera donc pas utile dans notre projet.

## 3. Baseline TF-IDF + Regression logistique

Le TF-IDF est un modèle qui mesure deux métriques : la fréquence d'apparition d'un terme dans un document (TF) et à quel point un terme est rare à travers tous les documents (IDF) — le résultat final est le produit des deux. Cette méthode est utile en analyse de sentiments car les mots spécifiques à certains contextes, peu fréquents globalement mais présents dans des phrases de sentiment particulier, auront un score élevé, ce qui les rend plus informatifs pour la classification. C'est un modèle simple et parfait à implémenter en premier lieu car il sert de référence quantitative pour évaluer si un modèle plus complexe comme FinBERT apporte un gain justifiant sa complexité.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorisation
vectorizer = TfidfVectorizer() 

X_train = vectorizer.fit_transform(df_train["text"])
X_test = vectorizer.transform(df_test["text"])

# Labels
y_train = df_train["label"]
y_test = df_test["label"]

print(X_train.shape)
print(X_test.shape)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
# Régression logistique
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)
# Prédictions
y_pred = model.predict(X_test)

# évaluation avec plusieurs métriques
print(classification_report(y_test, y_pred))

On constate un recall faible au niveau de la classe des négatifs, preuve que le modèle a du mal à distinguer les phrases négatives. Ceci peut s'expliquer par le fait que le TF-IDF par défaut utilise des unigrammes pour traiter chaque mot : l'expression "not good" par exemple est séparée en "not" et "good", et à ses yeux le "good" traduit le positif il passe donc à côté de la négation introduite par le "not". Ce recall faible sur la classe des négatifs est problématique d'un point de vue financier car cela signifie que soit on l'a confondu avec un positif et dans ce cas on risque d'investir sur un actif perdant, soit on l'a mis en neutre et dans ce cas on peut potentiellement ne pas liquider nos positions à temps sur un investissement perdant.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorisation en utilisant des bigrammes
vectorizer = TfidfVectorizer(ngram_range=(1,2))

X_train = vectorizer.fit_transform(df_train["text"])
X_test = vectorizer.transform(df_test["text"])

# Labels
y_train = df_train["label"]
y_test = df_test["label"]

print(X_train.shape)
print(X_test.shape)

# Régression logistique
# nos données sont fortement dominées par les neutres on rajoute l'argument class_weight='balanced'
# pour retirer le biais vers la classe majoritaire pendant l'entrainement
model = LogisticRegression(max_iter=1000, class_weight='balanced') 

model.fit(X_train, y_train)
# Prédictions
y_pred = model.predict(X_test)

# évaluation avec plusieurs métriques
print(classification_report(y_test, y_pred))

Le rajout du paramètre bigramme à notre TF-IDF a permis de capturer deux mots consécutifs et donc de préserver les négations, contrairement aux unigrammes (la séquence "not good" est maintenant vue comme une expression à part entière). Cette simple modification a permis un gain de +0.21 sur le recall des négatifs. Néanmoins le TF-IDF est un modèle limité qui ne cherche pas à comprendre le sens et le contexte des phrases. Il représente les séquences comme des sacs de mots, on perd ainsi l'information sur l'ordre des mots ainsi que sur le contexte. C'est pourquoi on va se pencher sur un modèle plus complexe comme FinBERT, basé sur les Transformers.

## 4. Préprocessing de FinBERT

In [ ]:
print(df_train.features)

# conversion du type label de value à classlabel
from datasets import ClassLabel

class_label = ClassLabel(names=["negative", "neutral", "positive"])

df_train = df_train.cast_column("label", class_label)
print(df_train.features)


In [ ]:
# split train validation
train_valid = df_train.train_test_split(
    test_size=0.2,
    seed=42,
    stratify_by_column="label"
)

train_dataset = train_valid["train"]
valid_dataset = train_valid["test"]

In [ ]:
# tokenisation des phrases pour pouvoir les passer à notre modele

from transformers import DataCollatorWithPadding
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
    )

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

valid_dataset = valid_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = df_test.map(
    tokenize_function,
    batched=True
)

data_collator = DataCollatorWithPadding(tokenizer)



La tokenisation est faite avec un padding dynamique pour que les sequences de chaque batch soit paddé avec la plus longue sequences du batch plutot que celle de tout le dataset ca permet de reduire l'utilisation de la memoire 

## 5. Fine-Tuning de FinBERT avec l'API Trainer

In [ ]:
# fine-tuning du modele Finbert
from transformers import TrainingArguments

training_args = TrainingArguments(
    "test-trainer",
    eval_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=2e-5
)

In [ ]:
from transformers import AutoModelForSequenceClassification
import evaluate

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=3)

def compute_metrics(eval_preds):
    f1_metric = evaluate.load("f1")
    recall_metric = evaluate.load("recall")

    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    f1_score = f1_metric.compute(predictions=predictions, references=labels, average="macro")
    recall_score = recall_metric.compute(predictions=predictions, references=labels, average="macro")
    return {"f1": f1_score["f1"], "recall": recall_score["recall"]}

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
test_results = trainer.evaluate(test_dataset)
print(test_results)

## 6. Baseline vs. Fine-Tuned FinBERT comparaison

| Model | Vectorisation | F1-Macro | Notes |
|---|---|---|---|
| Regression logistique | TF-IDF unigrams | ~0.79 | défaut |
| Regression logistique | TF-IDF bigrams, balanced | **0.82** | meilleur baseline |
| **FinBERT fine-tuné** | BERT tokenizer (contextual) | **0.934** | 3 epochs, lr=2e-5 |

Le modèle FinBERT fine-tuné offre une augmentation de +11.4 points sur la F1 Macro comparé à notre meilleur baseline TF-IDF. Ce gain illustre bien l'importance du contextual embedding les mots ont une représentation différente selon leur contexte et du pré-entraînement de FinBERT sur des rapports financiers (10-K filings), qui lui confère des connaissances du vocabulaire financier que le sac de mots ne peut capturer.

## 7. Analyse des résidus 

In [ ]:
predictions_output = trainer.predict(test_dataset)

# Extract logits and labels
logits = predictions_output.predictions
true_labels = predictions_output.label_ids

# Convert logits to predicted labels
predicted_labels = np.argmax(logits, axis=-1)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


class_names = df_train.features['label'].names

# Matrice de confusion pour analyser les erreurs du modele
cm = confusion_matrix(true_labels, predicted_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# on trouve les indices ou les predictions ne correspondent pas aux labels
misclassified_indices = np.where(predicted_labels != true_labels)[0]

# affichage de quelques erreurs du modele
print(f"Nombre de phrases mal classées : {len(misclassified_indices)}")
print("Quelques exemples de phrases mal classées :")

for i, idx in enumerate(misclassified_indices[:10]):
    original_text = test_dataset['text'][int(idx)]
    true_label_name = class_names[true_labels[idx]]
    predicted_label_name = class_names[predicted_labels[idx]]
    print(f"\nPhrase: {original_text}")
    print(f"Vraie étiquette: {true_label_name}")
    print(f"Étiquette prédite: {predicted_label_name}")


La matrice de confusion révèle que FinBERT fait encore des erreurs, et la majorité d'entre elles provient du fait qu'il prédit négatif sur des phrases positives (18 cas). En observant de plus près certaines de ces phrases, elles peuvent parfois présenter des ambiguïtés qui rendent la classification plus complexe par exemple : "Net interest income totaled EUR 15.9 mn, compared to EUR 15.6 mn a year earlier" et "Profit for the period was EUR 15.6 mn compared to EUR 14.1 mn in 2007" étaient étiquetées positives, le modele a prédit negatif,  mais si on regarde sous un autre angle les entreprises en question on connu une faible augmentation au fil des années. Une entreprises qui stagne n'est pas forcément un investissement intéressant. Toutefois, bien que le modèle fasse encore des erreurs, son recall sur la classe négative reste élevé ce qui signifie qu'il détecte bien les vrais signaux négatifs, là où le risque financier est le plus élevé. 